# 06 — GNN path planning: learning where water goes on the street graph

The drain planner routes with a **hand-set** cost (`length + 2400·climb`) and the
evacuation view needs the raster emulator re-run for every rainfall. This notebook builds
the learned replacement — **FloodGNN** — from first principles:

1. streets → a graph with *local* topographic/urban features (nothing city-specific),
2. message passing written by hand (gather + `index_add_` — that is all a GNN is),
3. FiLM conditioning so ONE model answers every storm size,
4. labels straight from our own twin (emulator depth sweeps + drain-planner flow),
5. train → evaluate → **route around the flood**.

Why it can matter beyond the dashboard: features are local and per-graph standardized, so
the model *transfers* — train on flat Patna, ask about hilly Bengaluru. That extends the
paper's DEM-uncertainty finding (flat cities don't co-locate, hilly ones do) to learned
models. The full experiment suite lives in `scripts/run_gnn_colab.py`; this notebook is
the readable walkthrough of the same code paths.

In [ ]:
import os
if not os.path.exists('project-varuna') and not os.path.exists('varuna'):
    !git clone -b feature/gnn-path-planning https://github.com/Maverick-Ansh/project-varuna.git
if os.path.exists('project-varuna'):
    %cd project-varuna
!pip -q install rasterio
import torch, numpy as np
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE)

## 1 — The street graph, with features a model can transfer

`varuna.gnn.graph.build_graph` reuses the *same* `roadnet.prepare` the drain planner uses
(same node order — labels index straight in) and attaches 11 features per node: local
elevation (standardized **per graph**, so 50 m Patna and 900 m Bengaluru look alike),
pit-ness (below-neighbours depth in metres), terrain grade, log flow-accumulation,
permanent water, SAR waterlogging frequency, building/road density, built-up flag, degree,
distance to the domain edge. Edges carry length, signed climb, and grade.

In [ ]:
from varuna.gnn.graph import build_graph, NODE_FEATURES
sg = build_graph('artifacts/patna')
print(f'{sg.n:,} street nodes, {sg.e:,} directed edges')
for i, name in enumerate(NODE_FEATURES):
    col = sg.x[:, i]
    print(f'  {name:14s} mean {col.mean():7.3f}   max {col.max():7.3f}')

In [ ]:
# the graph is the city: streets scatter-plotted, colored by elevation
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 6.5))
s = ax.scatter(sg.lon, sg.lat, c=sg.z, s=0.5, cmap='terrain')
plt.colorbar(s, label='elevation (m)'); ax.set_title('Patna street nodes')
ax.set_aspect(1 / np.cos(np.radians(sg.lat.mean())))
plt.savefig('street_graph_patna.png', dpi=120); plt.show()

## 2 — Message passing by hand (no library, no magic)

One round of message passing on a 5-node toy graph, every tensor printed. This is the
exact op `varuna/gnn/model.py` runs — a GNN layer is just:

```
m_uv = MLP([h_u, h_v, e_uv])        # a message per directed edge  (gather)
a_v  = mean/max of incoming m_uv    # aggregate at each node       (scatter)
h_v  = LayerNorm(h_v + MLP([h_v, a_v]))   # update, residual
```

Water flows downhill along edges; K rounds let a node "feel" terrain K hops away —
exactly the neighbourhood a pond spreads over.

In [ ]:
# toy: a Y-junction draining into node 4     0   1
#                                             \\ /
#                                              2 - 3 - 4
edge_index = torch.tensor([[0, 1, 2, 3],      # src
                           [2, 2, 3, 4]])     # dst  (directed: toward node 4)
h = torch.eye(5)[:, :3]                       # 5 nodes, 3-dim states (one-hot-ish)
print('states h:\n', h)
src, dst = edge_index
messages = h[src]                             # GATHER: each edge reads its source state
print('\nmessages (h[src]):\n', messages)
agg = torch.zeros(5, 3)
agg.index_add_(0, dst, messages)              # SCATTER: sum messages at each destination
print('\naggregated at dst (node 2 got BOTH 0 and 1):\n', agg)
deg = torch.zeros(5).index_add_(0, dst, torch.ones(4))
print('\nin-degree:', deg.tolist(), ' -> mean =', (agg / deg.clamp(min=1)[:, None])[2].tolist())

That is the whole trick. The real model wraps it with MLPs, a max-aggregation channel,
residuals + LayerNorm, and **FiLM**: a tiny net maps rainfall → per-layer `(gamma, beta)`
that scale/shift every node state, so the same weights answer 20 mm and 240 mm.

In [ ]:
from varuna.gnn.model import FloodGNN
m = FloodGNN(hidden=32, layers=2)
out = {r: m(sg.x[:5000], sg.edge_index[:, (sg.edge_index < 5000).all(0)],
            sg.edge_attr[(sg.edge_index < 5000).all(0)], r)['node_depth']
       for r in (40.0, 200.0)}
print('untrained, but rainfall already changes the answer:',
      float((out[200.0] - out[40.0]).abs().mean()))
print('parameters:', sum(p.numel() for p in m.parameters()))

## 3 — Labels from our own twin (no hand labelling, ever)

* **depth**: the U-Net emulator's max-depth grid for 12 storms (20…240 mm), sampled at
  every node cell and along every edge (ends + midpoint).
* **flow**: run the street drain planner at 3 (rain, inlets) configs and count the m³
  each node conveys — the GNN's flow head learns *where the trunk drains belong*.

~10 s per area on CPU, cached as `gnn_data.npz` in the bundle.

In [ ]:
from varuna.gnn.dataset import build_dataset
from varuna.areas import list_areas, is_built
for a in list_areas():
    w = a.work_dir()
    if is_built(a.id) and os.path.exists(f'{w}/road_graph.json.gz') \
            and not os.path.exists(f'{w}/gnn_data.npz'):
        print(a.id, '->', build_dataset(work=w, device=DEVICE)['wet_node_frac_100mm'])

## 4 — Train

Two lessons this project already paid for are baked in: floods are sparse, so wet targets
get ×20 loss weight (the first Patna emulator predicted zero everywhere and looked fine on
MSE), and validation holds out **storm sizes**, not random nodes — the slider asks the
model to interpolate rainfall it never saw.

In [ ]:
from varuna.gnn.train import train_gnn
works = [a.work_dir() for a in list_areas()
         if is_built(a.id) and os.path.exists(f'{a.work_dir()}/gnn_data.npz')]
ckpt, hist = train_gnn(works, out='artifacts/gnn/gnn.pt', hidden=96, layers=4,
                       epochs=80, device=DEVICE, amp=(DEVICE == 'cuda'), log_fn=print)

## 5 — Evaluate: accuracy, planning quality, speed, transfer

The report answers four questions — can it *rank* flooded streets (AUC), is the depth
right where wet (RMSE), does routing on GNN risk ≈ routing on oracle risk (wet metres per
route), and how fast vs the emulator pipeline. The transfer runs (train with a whole city
held out) are the paper's headline — run those via the script, they retrain twice.

In [ ]:
from varuna.gnn.evaluate import evaluate_model
import json
rep = evaluate_model('artifacts/gnn/gnn.pt', works, out='artifacts/gnn/gnn_report.json')
for area, v in rep['areas'].items():
    print(area, json.dumps({k: v[k] for k in ('edge_auc', 'wet_rmse_m', 'flow_spearman',
                                              'ms_per_query_gnn', 'ms_per_query_emulator')}))
print('\nrouting (patna):', json.dumps(rep['areas'].get('patna', {}).get('routing', {}), indent=1))

In [ ]:
# the payoff: route two points through a 120 mm storm, GNN-scored streets
from varuna.gnn.planner import safe_route
r = safe_route([25.625, 85.11], [25.585, 85.17], rain_mm=120, work='artifacts/patna')
print('backend:', r['backend'], '| detour +%s%%' % r['detour_pct'],
      '| wet street crossed:', r['route']['wet_length_m'], 'm',
      '(shortest would cross', r['shortest']['wet_length_m'], 'm)')

## 6 — Full suite + push

```
!python scripts/run_gnn_colab.py --ablation --transfer --figures --device cuda
!python scripts/run_gnn_colab.py --push --branch feature/gnn-path-planning
```

(`GITHUB_TOKEN` as a Colab/Kaggle secret; see RUNBOOK_GNN.md for the merge + deploy steps —
once `artifacts/gnn/gnn.pt` reaches main and the Space redeploys, the dashboard's route
panel badge flips from *emulator* to *GNN*.)